# 01 — Exploratory Data Analysis (EDA)

**Tujuan:** memahami data sebelum memilih model, yaitu distribusi kelas, ukuran dan kualitas gambar, serta jumlah teks per kelas (jika output OCR tersedia).

**Cara menjalankan di Kaggle:**
- Input: data kompetisi. Opsional: output notebook `04_ocr_doctr` (untuk bagian statistik OCR).
- Accelerator: **None** (cukup CPU). Perkiraan waktu sekitar 5–10 menit.

**Output:** grafik di `/kaggle/working/figures/` (salin ke `reports/figures/` di repo).

In [ ]:
import os
from multiprocessing import Pool
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageStat
from tqdm.auto import tqdm

FIG = Path("/kaggle/working/figures")
FIG.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = os.cpu_count() or 2
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Membaca metadata
Path di `train.csv` tertulis `train/<kelas>/x.tif`, sedangkan foldernya setelah diekstrak menjadi `train/train/<kelas>/x.tif`. Fungsi `resolve` menangani kedua bentuk itu.

In [ ]:
ROOT = sorted(Path("/kaggle/input").rglob("train.csv"))[0].parent
train_df = pd.read_csv(ROOT / "train.csv")
test_df = pd.read_csv(ROOT / "test.csv")

def resolve(rel):
    """Mengembalikan path file yang benar-benar ada untuk path relatif dari CSV."""
    rel = str(rel)
    for cand in (ROOT / rel, ROOT / Path(rel).parts[0] / rel):
        if cand.exists():
            return str(cand)
    return None

train_df["path"] = [resolve(f) for f in train_df["filename"]]
test_df["path"] = [resolve(f) for f in test_df["filename"]]
print(f"Train: {len(train_df)} halaman | Test: {len(test_df)} halaman | kelas: {train_df['class_name'].nunique()}")
print("File tidak ditemukan - train:", train_df["path"].isna().sum(), "| test:", test_df["path"].isna().sum())
train_df.head()

## 2. Distribusi kelas

In [ ]:
counts = train_df["class_name"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(counts.index, counts.values, color="#4C72B0")
ax.set_xlabel("Jumlah halaman")
ax.set_title("Distribusi kelas pada data train")
for i, v in enumerate(counts.values):
    ax.text(v + 10, i, str(v), va="center", fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "class_distribution.png"); plt.show()
print(f"Min {counts.min()} | max {counts.max()} | rasio max/min {counts.max()/counts.min():.3f}")

**Observasi:** data sangat seimbang (sekitar 2.000 halaman per kelas), jadi akurasi adalah metrik yang adil dan tidak perlu class weighting atau oversampling.

## 3. Audit kualitas gambar: bisa dibaca, ukuran, dan kontras

In [ ]:
def audit_one(path):
    """Memeriksa satu gambar: bisa dibuka, ukuran, rata-rata dan std piksel."""
    try:
        with Image.open(path) as im:
            im.seek(0)
            w, h, mode = im.size[0], im.size[1], im.mode
            g = im.convert("L"); g.thumbnail((256, 256))
            st = ImageStat.Stat(g)
            return path, True, w, h, mode, st.mean[0], st.stddev[0]
    except Exception:
        return path, False, 0, 0, "", np.nan, np.nan

paths = train_df["path"].dropna().tolist() + test_df["path"].tolist()
with Pool(NUM_WORKERS) as pool:
    rows = list(tqdm(pool.imap(audit_one, paths, chunksize=64), total=len(paths)))
audit = pd.DataFrame(rows, columns=["path", "ok", "w", "h", "mode", "mean", "std"])
audit["split"] = ["train"] * train_df["path"].notna().sum() + ["test"] * len(test_df)

print("File rusak (tidak bisa dibuka):", int((~audit["ok"]).sum()))
print("Halaman benar-benar kosong (std < 1):", int((audit["std"] < 1).sum()))
print("Mode gambar:", audit.loc[audit.ok, "mode"].value_counts().to_dict())
print(audit.loc[audit.ok, ["w", "h"]].describe().round(0).to_string())

In [ ]:
ok = audit[audit.ok]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(ok["h"], bins=40, color="#4C72B0"); axes[0].set_title("Tinggi gambar (px)")
axes[1].hist(ok["w"], bins=40, color="#55A868"); axes[1].set_title("Lebar gambar (px)")
plt.tight_layout(); plt.savefig(FIG / "image_sizes.png"); plt.show()

**Observasi:** hampir semua halaman memiliki tinggi 1.000 px dengan lebar sekitar 750 px (resolusi scan rendah, sekitar 100 dpi). Artinya resolusi lebih dari 1024x768 tidak menambah detail, hanya memperbesar gambar.

## 4. Kepadatan "tinta" per kelas
Rata-rata piksel yang rendah berarti halaman lebih gelap/padat (banyak teks, gambar, atau border hitam). Std yang rendah berarti halaman hampir kosong.

In [ ]:
m = ok.merge(train_df[["path", "class_name"]], on="path")
order = m.groupby("class_name")["mean"].median().sort_values().index
fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot([m.loc[m.class_name == c, "mean"] for c in order], vert=False, showfliers=False)
ax.set_yticks(range(1, len(order) + 1))
ax.set_yticklabels(order)
ax.set_xlabel("Rata-rata intensitas piksel (0 = hitam, 255 = putih)")
ax.set_title("Kepadatan halaman per kelas")
plt.tight_layout(); plt.savefig(FIG / "ink_density.png"); plt.show()

## 5. Contoh halaman per kelas
Bagian ini memperlihatkan kondisi kualitas scan: border hitam, halaman lain yang ikut terbawa, teks samar, tulisan miring, dan halaman yang hampir kosong.

In [ ]:
classes = sorted(train_df["class_name"].unique())
good = set(audit.loc[audit.ok, "path"])              # lewati file rusak
fig, axes = plt.subplots(4, 8, figsize=(16, 11))
for ax, (cls, k) in zip(axes.ravel(), [(c, j) for c in classes for j in range(2)]):
    p =[x for x in train_df[train_df.class_name == cls]["path"].dropna() if x in good][k * 7]
    with Image.open(p) as im:
        ax.imshow(im.convert("L"), cmap="gray")
    ax.set_title(cls, fontsize=8); ax.axis("off")
plt.suptitle("Dua contoh halaman per kelas", y=1.0)
plt.tight_layout(); plt.savefig(FIG / "samples_per_class.png"); plt.show()

## 6. Statistik teks hasil OCR (jika output docTR dilampirkan)

In [ ]:
ocr_files = sorted(Path("/kaggle/input").rglob("ocr_text.csv"))
if ocr_files:
    ocr = pd.read_csv(ocr_files[0])
    q = ocr.merge(train_df[["id", "class_name"]], on="id")
    summ = q.groupby("class_name").agg(median_words=("n_words", "median"),
                                       pct_under_5=("n_words", lambda s: 100 * (s < 5).mean()),
                                       mean_conf=("mean_conf", "mean")).sort_values("median_words")
    print(summ.round(2).to_string())
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(summ.index, summ["median_words"], color="#C44E52")
    ax.set_xscale("log"); ax.set_xlabel("Median jumlah kata OCR per halaman (skala log)")
    ax.set_title("Jumlah teks per kelas")
    plt.tight_layout(); plt.savefig(FIG / "ocr_words_per_class.png"); plt.show()
else:
    print("Output OCR tidak dilampirkan, bagian ini dilewati.")

## 7. Kesimpulan EDA

1. **Kelas seimbang**, sehingga akurasi bisa dipakai sebagai metrik utama.
2. **Gambar grayscale/1-bit beresolusi rendah** (sekitar 1000x760 px). Downsampling harus antialiased supaya teks tipis tidak hilang.
3. **Kualitas scan buruk tetapi konsisten** antara train dan test, sehingga gambar **tidak dibersihkan**. Model harus belajar dari kondisi yang sama dengan data test.
4. **Hanya 1 file rusak** dan tidak ada halaman yang benar-benar kosong, sehingga pembersihan data cukup membuang file rusak itu.
5. **Jumlah teks sangat berbeda antar kelas** (file_folder hampir kosong, scientific_publication sangat padat). Ini alasan model teks dari OCR bisa menjadi pelengkap model gambar.